<a href="https://colab.research.google.com/github/SAJLENDRAPANDEY/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# My Rule

The purpose of this baseline is to identify webpages that should be reviewed for a possible content refresh.

The rule uses three observable signals:

- Days since the page was last updated
- Search impressions during the last 90 days
- Click-through rate (CTR)

Pages that are older, receive many impressions, and have a low CTR receive higher scores because improving these pages may have a greater impact than updating pages with little search visibility.

This baseline is transparent, easy to explain, and intended to support SEO decisions rather than replace human judgment.

## Reason Codes

- STALE_CONTENT → Page has not been updated for 180 days or more.
- HIGH_VISIBILITY → Page has at least 3,000 search impressions in the last 90 days.
- LOW_CTR → Page CTR is below 1%.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Baseline score
df["score"] = 0

# Stale content
df.loc[df["days_since_last_update"] >= 180, "score"] += 3

# High visibility
df.loc[df["impressions_90d"] >= 3000, "score"] += 2

# Low CTR
df.loc[df["ctr"] < 1.0, "score"] += 2


# Reason codes
def reason(row):

    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["impressions_90d"] >= 3000:
        reasons.append("HIGH_VISIBILITY")

    if row["ctr"] < 1.0:
        reasons.append("LOW_CTR")

    return ", ".join(reasons)


df["reason_code"] = df.apply(reason, axis=1)


# Action label
def action(score):

    if score >= 6:
        return "Refresh Now"

    elif score >= 3:
        return "Review"

    else:
        return "Monitor"


df["action"] = df["score"].apply(action)


# Rank pages
df = df.sort_values("score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1


# Save CSV
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action,rank
0,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,43.33,0.0,good,page_3_5,down,-74.7,7,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now,1
1,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,40.00,0.0,good,page_3_5,down,-89.1,7,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now,2
2,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,42.99,0.0,excellent,page_3_5,down,-81.5,7,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now,3
3,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,24.11,0.0,excellent,striking,down,-85.6,7,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now,4
4,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,41.76,0.0,good,striking,down,-51.8,7,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now,5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# Top-20 Review

The highest-ranked pages were reviewed using the baseline score.

For each page:

- **Action** was assigned from the baseline score.
- **Reason Code** explains why the page received its score.
- **Confidence Note**
  - High: Multiple signals support refreshing the page.
  - Medium: Some signals support review.
  - Low: Limited evidence; monitoring may be enough.
- **What Would Make It Wrong**
  - Search demand may have changed recently.
  - A recent update may not yet appear in the data.
  - Low CTR may be caused by search intent rather than page quality.
  - Some pages may have low business importance despite a high score.

In [25]:
def confidence(score):

    if score >= 6:
        return "High"

    elif score >= 3:
        return "Medium"

    else:
        return "Low"


def review_note(row):

    if row["score"] >= 6:
        return "Recommendation could be wrong if recent updates or search intent changes are not reflected in the dataset."

    elif row["score"] >= 3:
        return "Manual review is recommended before refreshing."

    else:
        return "Current evidence suggests monitoring rather than immediate action."


top20 = df.head(20).copy()

top20["confidence"] = top20["score"].apply(confidence)

top20["what_would_make_it_wrong"] = top20.apply(
    review_note,
    axis=1
)

top20[
[
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "confidence",
    "what_would_make_it_wrong"
]]

,rank,content_id,score,action,reason_code,confidence,what_would_make_it_wrong
0,1,content_1bfaa38ff26c,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
1,2,content_5feee3994adb,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
2,3,content_7368877ea310,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
3,4,content_cf56e2e2e282,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
4,5,content_0a91db491d14,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
5,6,content_fe16a55cd13d,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
6,7,content_ecb6215e79fd,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
7,8,content_c2d929d83eaa,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
8,9,content_b16bd7307b39,7,Refresh Now,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",High,Recommendation could be wrong if recent update...
9,10,content_107776820988,5,Review,"STALE_CONTENT, LOW_CTR",Medium,Manual review is recommended before refreshing.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak Picks

Some high-scoring pages may still be weak recommendations if they have very low search impressions or limited business value.

These pages should be manually reviewed before deciding to refresh them.

# Leakage Check

The baseline only uses information available before making a recommendation:

- Days since last update
- Search impressions (last 90 days)
- CTR

No future information, product flags, client names, URLs, or outcome variables were used.

Based on this review, no obvious data leakage was observed.

In [26]:
weak_picks = df[
    (df["score"] >= 6) &
    (df["impressions_90d"] < 300)
]

weak_picks[
[
    "content_id",
    "score",
    "impressions_90d",
    "ctr"
]]

,content_id,score,impressions_90d,ctr


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.